In [1]:
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
%pip install --upgrade transformers accelerate

Looking in indexes: https://download.pytorch.org/whl/cpu
Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig,get_peft_model,TaskType
from datasets  import load_dataset


W0304 09:58:32.143000 21452 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [5]:
model_name="TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [6]:
tokenizer=AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token=tokenizer.eos_token

In [7]:
model_path="./lora_trained_model/checkpoint-5"

In [8]:
non_instructed_trained_model=AutoModelForCausalLM.from_pretrained(model_path,device_map="auto")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.


WARN  Feature `utils/Perplexity` requires Python < 3.14 and Python GIL enabled and Python >= 3.13.3T (T for Threading-Free edition of Python) plus Torch 2.8. Feature is currently skipped/disabled.


INFO  ENV: Auto setting PYTORCH_ALLOC_CONF='expandable_segments:True,max_split_size_mb:256,garbage_collection_threshold:0.7' for memory saving.


INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.          


DEBUG BitBLAS import failed: No module named 'bitblas'                         


DEBUG Skipping qlinear module import `bitblas_target_detector`: No module named 'thefuzz'


INFO  

_____/\\\\\\\\\\\\__/\\\\\\\\\\\\\____/\\\\\\\\\\\\\\\______________________/\\\________/\\\\____________/\\\\_______________________/\\\__________________/\\\\\\____
 ___/\\\//////////__\/\\\/////////\\\_\///////\\\/////____________________/\\\\/\\\\____\/\\\\\\________/\\\\\\______________________\/\\\_________________\////\\\____
  __/\\\_____________\/\\\_______\/\\\_______\/\\\_______________________/\\\//\////\\\__\/\\\//\\\____/\\\//\\\______________________\/\\\____________________\/\\\____
   _\/\\\____/\\\\\\\_\/\\\\\\\\\\\\\/________\/\\\________/\\\\\\\\\\\__/\\\______\//\\\_\/\\\\///\\\/\\\/_\/\\\_____/\\\\\___________\/\\\______/\\\\\\\\_____\/\\\____
    _\/\\\___\/////\\\_\/\\\/////////__________\/\\\_______\///////////__\//\\\______/\\\__\/\\\__\///\\\/___\/\\\___/\\\///\\\____/\\\\\\\\\____/\\\/////\\\____\/\\\____
     _\/\\\_______\/\\\_\/\\\___________________\/\\\______________________\///\\\\/\\\\/___\/\\\____\///_____\/\\\__/\\\__\//\\\__/\\\////\\\___/\

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [9]:
prompt="Continuous learning is not limited to formal education. It includes"

In [10]:
inputs=tokenizer(prompt,return_tensors="pt")

In [12]:
outputs = non_instructed_trained_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

In [13]:

print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Model Output:

Continuous learning is not limited to formal education. It includes informal learning, as well as everyday life experiences.
What is the future of learning?
Learning is never-ending; there's always something new to learn and expand your knowledge. Whether it's a new skill you want to learn or just want to brush up on some old ones, online courses can help you grow in many different areas. You can learn about any topic from music to business – even the latest gadgets and apps! With a wide variety


In [15]:
dataset=load_dataset("Amod/mental_health_counseling_conversations",split="train")

INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/Amod/mental_health_counseling_conversations/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/Amod/mental_health_counseling_conversations/d7e86f0813c5690181b41f97403c3674aa55dcef/README.md "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/datasets/Amod/mental_health_counseling_conversations/d7e86f0813c5690181b41f97403c3674aa55dcef/README.md "HTTP/1.1 200 OK"


README.md: 0.00B [00:00, ?B/s]

c:\Users\INMOR14\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\INMOR14\.cache\huggingface\hub\datasets--Amod--mental_health_counseling_conversations. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/Am

combined_dataset.json: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [19]:
def format_row(example):
    question=example["Context"]
    answer=example["Response"]
    example["Text"]=f"[INST] {question} [/RESPONSE] {answer}"
    return example

In [28]:
formtted_dataset=dataset.map(format_row)
formtted_dataset

Map:   0%|          | 0/3512 [00:00<?, ? examples/s]

Dataset({
    features: ['Context', 'Response', 'Text'],
    num_rows: 3512
})

In [21]:
import pandas as pd

df=pd.DataFrame(dataset)

In [25]:
df.to_csv("mhsc.csv",index=False)
df.to_json("mhsc.jsonl",orient="records",lines=True)

In [29]:
dataset=load_dataset("csv",data_files="./pharma_instruction_data.csv",split="train")
dataset

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 5
})

In [30]:
def format_example(example):
    prompt = f"### Instruction:\n{example['instruction']}\n### Input:\n{example['input']}\n### Response:\n{example['output']}"
    return {"text": prompt}
     

In [31]:
dataset=dataset.map(format_example)

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

In [32]:
def tokenize_fn(example):
    tokens=tokenizer(example["text"],padding="max_length",truncation=True,max_length=512)
    tokens["labels"]=tokens["input_ids"].copy()
    return tokens

In [34]:
tokenized=dataset.map(tokenize_fn,batched=True)

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

In [35]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none"
)

In [36]:
instructed_model=get_peft_model(non_instructed_trained_model,lora_config)

c:\Users\INMOR14\AppData\Local\Programs\Python\Python311\Lib\site-packages\peft\mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
c:\Users\INMOR14\AppData\Local\Programs\Python\Python311\Lib\site-packages\peft\tuners\tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [50]:
training_args=TrainingArguments(
    output_dir="./instructed-model",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    logging_steps=20,
    gradient_accumulation_steps=8,
    fp16=True,
    learning_rate=2e-5,
    save_total_limit=1,
    report_to="none"
)

In [51]:
trainer=Trainer(
    instructed_model,
    training_args,
    train_dataset=tokenized
)

In [52]:
trainer.train()

Step,Training Loss


INFO:httpx:HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T/59f6f375b26bde864a6ca194a9a3044570490064/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T/59f6f375b26bde864a6ca194a9a3044570490064/config.json "HTTP/1.1 200 OK"


TrainOutput(global_step=1, training_loss=9.88018798828125, metrics={'train_runtime': 41.6228, 'train_samples_per_second': 0.12, 'train_steps_per_second': 0.024, 'total_flos': 15907411722240.0, 'train_loss': 9.88018798828125, 'epoch': 1.0})

In [55]:

model_path = "./instructed-model/checkpoint-1"

In [56]:
model=AutoModelForCausalLM.from_pretrained(model_path,device_map="auto")

INFO:httpx:HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T/59f6f375b26bde864a6ca194a9a3044570490064/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T/59f6f375b26bde864a6ca194a9a3044570490064/config.json "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T/59f6f375b26bde864a6ca194a9a3044570490064/generation_config.json "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [ ]:

prompt = "Explain the mechanism of action of Metformin."

In [ ]:
inputs=tokenizer(prompt,return_tensors="pt")

In [ ]:
outputs=model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

In [ ]:

print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Model Output:

Explain the mechanism of action of Metformin.
Explain the mechanism of action of Clopidogrel.
Asked 2 years, 10 months ago
Metformin is a thiazolidinedione which has the ability to inhibit the action of AMP kinase and insulin receptor. This results in increased production of ATP from glycolysis (glucose oxidation) and decreased production of lactic acid (oxidative phosphorylation). It also
